In [2]:
import pandas as pd
import numpy as np

from scipy.stats import wasserstein_distance

In [4]:
REFERENCE_CSV = (
    "../../data/processed/CTB/sampled_real_eventlogs/s6_sample_24.000_eventlog_target_rank_features.csv"
)

SIM_CSV = (
    "../../data/processed/CTB/prosit_simulations/sim_log_s6_sample_30.000_depth10.csv"
)

ref = pd.read_csv(REFERENCE_CSV)
sim = pd.read_csv(SIM_CSV)

print(
    f"Reference events: {len(ref):,}"
)

print(
    f"Simulation events: {len(sim):,}"
)

Reference events: 74,066
Simulation events: 75,158


In [6]:
# ==========================================================
# ACTIVITY COUNTS
# ==========================================================

ref_acts = (
    ref["concept:name"]
    .value_counts()
    .rename("reference")
)

sim_acts = (
    sim["concept:name"]
    .value_counts()
    .rename("simulation")
)

activity_comparison = pd.concat(
    [ref_acts, sim_acts],
    axis=1
).fillna(0)

activity_comparison["diff"] = (
    activity_comparison["simulation"]
    - activity_comparison["reference"]
)

activity_comparison["diff_pct"] = (
    100
    * activity_comparison["diff"]
    / activity_comparison["reference"]
)

activity_comparison.sort_index()

,reference,simulation,diff,diff_pct
Gate In,24000,24000,0,0.000000
Gate Out,24000,24000,0,0.000000
HO2_delivery,160,1043,883,551.875000
HO2_mixed,39,199,160,410.256410
HO2_receive,51,329,278,545.098039
LL_delivery,1206,3,-1203,-99.751244
LL_mixed,561,252,-309,-55.080214
LL_receive,49,1,-48,-97.959184
RMG_delivery,6150,7017,867,14.097561
RMG_mixed,4446,3105,-1341,-30.161943


In [7]:
# ==========================================================
# RESOURCE COUNTS
# ==========================================================

ref_res = (
    ref["org:resource"]
    .value_counts()
    .rename("reference")
)

sim_res = (
    sim["org:resource"]
    .value_counts()
    .rename("simulation")
)

resource_comparison = pd.concat(
    [ref_res, sim_res],
    axis=1
).fillna(0)

resource_comparison["diff"] = (
    resource_comparison["simulation"]
    - resource_comparison["reference"]
)

resource_comparison["diff_pct"] = (
    100
    * resource_comparison["diff"]
    / resource_comparison["reference"]
)

resource_comparison.sort_index()

,reference,simulation,diff,diff_pct
HO2,250,1571,1321,528.400000
LL,1816,256,-1560,-85.903084
Res.GateIn,24000,24000,0,0.000000
Res.GateOut,24000,24000,0,0.000000
T06,1081,1362,281,25.994450
T07,903,803,-100,-11.074197
T08,883,806,-77,-8.720272
T09,961,1030,69,7.180021
T10,1048,961,-87,-8.301527
T11,1082,1115,33,3.049908


In [8]:
# ==========================================================
# PROCESS TYPE COUNTS
# ==========================================================

rmg_types = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

ref_proc = (
    ref[
        ref["concept:name"].isin(rmg_types)
    ]["concept:name"]
    .value_counts()
    .rename("reference")
)

sim_proc = (
    sim[
        sim["concept:name"].isin(rmg_types)
    ]["concept:name"]
    .value_counts()
    .rename("simulation")
)

process_comparison = pd.concat(
    [ref_proc, sim_proc],
    axis=1
)

process_comparison["diff_pct"] = (
    100
    * (
        process_comparison["simulation"]
        - process_comparison["reference"]
    )
    / process_comparison["reference"]
)

process_comparison

,reference,simulation,diff_pct
RMG_receive,13404,15209,13.466130
RMG_delivery,6150,7017,14.097561
RMG_mixed,4446,3105,-30.161943


In [10]:
# ==========================================================
# CALCULATE SIMULATION KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    sim[col] = pd.to_datetime(sim[col])

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

sim["waiting_time"] = (
    sim["start:timestamp"]
    - sim["enabled:timestamp"]
).dt.total_seconds() / 60

sim["service_time"] = (
    sim["time:timestamp"]
    - sim["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

case_start = (
    sim.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

case_end = (
    sim.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

sim["turnaround_time"] = (
    case_end
    - case_start
).dt.total_seconds() / 60

# ----------------------------------------------------------
# RMG EVENTS ONLY
# ----------------------------------------------------------

rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

sim_rmg = sim[
    sim["concept:name"]
    .isin(rmg_activities)
].copy()

ref_rmg = ref[
    ref["concept:name"]
    .isin(rmg_activities)
].copy()

print(
    f"Reference RMG events: {len(ref_rmg):,}"
)

print(
    f"Simulation RMG events: {len(sim_rmg):,}"
)

Reference RMG events: 24,000
Simulation RMG events: 25,331


In [17]:
#========== CALCULTATE REF KPIS =================
# ==========================================================
# CALCULATE REFERENCE KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    ref[col] = pd.to_datetime(ref[col])

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

ref["waiting_time"] = (
    ref["start:timestamp"]
    - ref["enabled:timestamp"]
).dt.total_seconds() / 60

ref["service_time"] = (
    ref["time:timestamp"]
    - ref["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

ref_case_start = (
    ref.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

ref_case_end = (
    ref.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

ref["turnaround_time"] = (
    ref_case_end
    - ref_case_start
).dt.total_seconds() / 60

# ----------------------------------------------------------
# RMG EVENTS ONLY
# ----------------------------------------------------------

rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

ref_rmg = ref[
    ref["concept:name"]
    .isin(rmg_activities)
].copy()

print(
    f"Reference RMG events: {len(ref_rmg):,}"
)

print(
    f"Reference cases: "
    f"{ref['case:concept:name'].nunique():,}"
)

print(
    "\nKPI columns created:"
)

print(
    [
        "waiting_time",
        "service_time",
        "turnaround_time"
    ]
)

Reference RMG events: 24,000
Reference cases: 24,000

KPI columns created:
['waiting_time', 'service_time', 'turnaround_time']


In [20]:
#MAP RECEIVE; DELIVER & MIXED
def add_operation_type(df):

    df = df.copy()

    df["operation_type"] = np.where(
        df["concept:name"].str.contains(
            "receive",
            case=False,
            na=False
        ),
        "receive",
        np.where(
            df["concept:name"].str.contains(
                "delivery",
                case=False,
                na=False
            ),
            "delivery",
            np.where(
                df["concept:name"].str.contains(
                    "mixed",
                    case=False,
                    na=False
                ),
                "mixed",
                np.nan
            )
        )
    )

    return df

ref = add_operation_type(ref)
sim = add_operation_type(sim)

In [21]:
#COMPARING FUNCTION'
def evaluate_kpi(
    ref_values,
    sim_values,
    kpi_name,
    segment
):

    ref_values = (
        pd.Series(ref_values)
        .dropna()
    )

    sim_values = (
        pd.Series(sim_values)
        .dropna()
    )

    return {
        "segment": segment,
        "kpi": kpi_name,

        "ref_mean":
        ref_values.mean(),

        "sim_mean":
        sim_values.mean(),

        "ref_median":
        ref_values.median(),

        "sim_median":
        sim_values.median(),

        "ref_std":
        ref_values.std(),

        "sim_std":
        sim_values.std(),

        "ref_p95":
        ref_values.quantile(0.95),

        "sim_p95":
        sim_values.quantile(0.95),

        "wasserstein":
        wasserstein_distance(
            ref_values,
            sim_values
        )
    }

In [22]:
# ==========================================================
# CASE LEVEL TURNAROUND VALIDATION
# ==========================================================

ref_cases = (
    ref.groupby(
        "case:concept:name"
    )["turnaround_time"]
    .max()
)

sim_cases = (
    sim.groupby(
        "case:concept:name"
    )["turnaround_time"]
    .max()
)

case_eval = pd.DataFrame(
    [
        {
            "kpi": "turnaround_time",

            "ref_mean":
            ref_cases.mean(),

            "sim_mean":
            sim_cases.mean(),

            "ref_median":
            ref_cases.median(),

            "sim_median":
            sim_cases.median(),

            "wasserstein":
            wasserstein_distance(
                ref_cases,
                sim_cases
            )
        }
    ]
)

case_eval

,kpi,ref_mean,sim_mean,ref_median,sim_median,wasserstein
0,turnaround_time,37.307083,61.382796,30.0,28.0,29.094596


In [23]:
# WHOLE EVALUATION
results = []
segments = {

    "all_rmg": (
        ref_rmg,
        sim_rmg
    ),

    "receive": (
        ref_rmg[
            ref_rmg["concept:name"]
            == "RMG_receive"
        ],
        sim_rmg[
            sim_rmg["concept:name"]
            == "RMG_receive"
        ]
    ),

    "delivery": (
        ref_rmg[
            ref_rmg["concept:name"]
            == "RMG_delivery"
        ],
        sim_rmg[
            sim_rmg["concept:name"]
            == "RMG_delivery"
        ]
    ),

    "mixed": (
        ref_rmg[
            ref_rmg["concept:name"]
            == "RMG_mixed"
        ],
        sim_rmg[
            sim_rmg["concept:name"]
            == "RMG_mixed"
        ]
    )
}

for segment, (
    ref_seg,
    sim_seg
) in segments.items():

    for kpi in [
        "waiting_time",
        "service_time",
        "turnaround_time"
    ]:

        results.append(

            evaluate_kpi(
                ref_seg[kpi],
                sim_seg[kpi],
                kpi,
                segment
            )

        )

evaluation = pd.DataFrame(
    results
)

In [24]:
# FINAL EVALUATION TABLE
evaluation = evaluation.round(2)

evaluation = evaluation.sort_values(
    [
        "segment",
        "kpi"
    ]
)

evaluation

,segment,kpi,ref_mean,sim_mean,ref_median,sim_median,ref_std,sim_std,ref_p95,sim_p95,wasserstein
1,all_rmg,service_time,11.65,16.01,8.0,9.0,11.33,107.16,31.00,30.00,4.96
2,all_rmg,turnaround_time,37.31,75.32,30.0,34.0,27.76,291.50,85.00,106.00,38.04
0,all_rmg,waiting_time,7.32,29.74,3.0,6.0,14.23,218.46,32.00,37.00,22.43
7,delivery,service_time,8.50,10.37,6.0,7.0,8.62,67.80,23.00,21.00,2.67
8,delivery,turnaround_time,29.91,73.92,23.0,34.0,25.52,288.36,69.00,99.17,44.03
6,delivery,waiting_time,6.17,30.93,3.0,6.0,9.27,225.31,25.00,36.00,24.76
10,mixed,service_time,14.56,20.64,11.0,11.0,12.95,120.64,37.00,42.80,6.28
11,mixed,turnaround_time,40.88,76.83,34.0,36.0,28.42,281.57,90.00,113.03,35.98
9,mixed,waiting_time,7.99,26.49,3.0,6.0,15.34,201.02,33.75,36.00,18.50
4,receive,service_time,12.13,17.67,9.0,9.0,11.52,118.38,32.00,31.00,5.89


In [25]:
# QUICK OVERVIEW ONLY WASSERSETEIN
evaluation.pivot(
    index="segment",
    columns="kpi",
    values="wasserstein"
).round(2)

kpi,service_time,turnaround_time,waiting_time
segment,,,
all_rmg,4.96,38.04,22.43
delivery,2.67,44.03,24.76
mixed,6.28,35.98,18.50
receive,5.89,36.33,22.24


COMPARING STRUCTURAL PATTERNS

In [26]:
# ==========================================================
# ACTIVITY COUNTS
# ==========================================================

ref_acts = (
    ref["concept:name"]
    .value_counts()
    .rename("reference")
)

sim_acts = (
    sim["concept:name"]
    .value_counts()
    .rename("simulation")
)

activity_comparison = pd.concat(
    [ref_acts, sim_acts],
    axis=1
).fillna(0)

activity_comparison["diff"] = (
    activity_comparison["simulation"]
    - activity_comparison["reference"]
)

activity_comparison["diff_pct"] = (
    100
    * activity_comparison["diff"]
    / activity_comparison["reference"]
)

activity_comparison.sort_index()

,reference,simulation,diff,diff_pct
Gate In,24000,24000,0,0.000000
Gate Out,24000,24000,0,0.000000
HO2_delivery,160,1043,883,551.875000
HO2_mixed,39,199,160,410.256410
HO2_receive,51,329,278,545.098039
LL_delivery,1206,3,-1203,-99.751244
LL_mixed,561,252,-309,-55.080214
LL_receive,49,1,-48,-97.959184
RMG_delivery,6150,7017,867,14.097561
RMG_mixed,4446,3105,-1341,-30.161943


In [27]:
# ==========================================================
# RESOURCE DISTRIBUTIONS COUNTS
# ==========================================================

ref_res = (
    ref["org:resource"]
    .value_counts()
    .rename("reference")
)

sim_res = (
    sim["org:resource"]
    .value_counts()
    .rename("simulation")
)

resource_comparison = pd.concat(
    [ref_res, sim_res],
    axis=1
).fillna(0)

resource_comparison["diff"] = (
    resource_comparison["simulation"]
    - resource_comparison["reference"]
)

resource_comparison["diff_pct"] = (
    100
    * resource_comparison["diff"]
    / resource_comparison["reference"]
)

resource_comparison.sort_index()

,reference,simulation,diff,diff_pct
HO2,250,1571,1321,528.400000
LL,1816,256,-1560,-85.903084
Res.GateIn,24000,24000,0,0.000000
Res.GateOut,24000,24000,0,0.000000
T06,1081,1362,281,25.994450
T07,903,803,-100,-11.074197
T08,883,806,-77,-8.720272
T09,961,1030,69,7.180021
T10,1048,961,-87,-8.301527
T11,1082,1115,33,3.049908


In [19]:
# ==========================================================
# PROCESS TYPE COUNTS
# ==========================================================

rmg_types = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

ref_proc = (
    ref[
        ref["concept:name"].isin(rmg_types)
    ]["concept:name"]
    .value_counts()
    .rename("reference")
)

sim_proc = (
    sim[
        sim["concept:name"].isin(rmg_types)
    ]["concept:name"]
    .value_counts()
    .rename("simulation")
)

process_comparison = pd.concat(
    [ref_proc, sim_proc],
    axis=1
)

process_comparison["diff_pct"] = (
    100
    * (
        process_comparison["simulation"]
        - process_comparison["reference"]
    )
    / process_comparison["reference"]
)

process_comparison

,reference,simulation,diff_pct
RMG_receive,11221,11957,6.559130
RMG_delivery,5112,1412,-72.378717
RMG_mixed,3667,2702,-26.315789


In [20]:
#===========================
# MEAN ERROR
#===========================
evaluation["mean_error_pct"] = (
    100
    * (
        evaluation["sim_mean"]
        - evaluation["ref_mean"]
    )
    / evaluation["ref_mean"]
)

evaluation[
    [
        "segment",
        "kpi",
        "mean_error_pct"
    ]
]

,segment,kpi,mean_error_pct
1,all_rmg,service_time,38.770282
2,all_rmg,turnaround_time,49.063670
0,all_rmg,waiting_time,253.021978
7,delivery,service_time,48.760331
8,delivery,turnaround_time,95.250926
6,delivery,waiting_time,432.730263
10,mixed,service_time,58.277254
11,mixed,turnaround_time,55.179137
9,mixed,waiting_time,233.626098
4,receive,service_time,23.786008


In [21]:
#================
# MEDIAN ERROR 
#==========================

evaluation["median_error_pct"] = (
    100
    * (
        evaluation["sim_median"]
        - evaluation["ref_median"]
    )
    / evaluation["ref_median"]
)

evaluation[
    [
        "segment",
        "kpi",
        "median_error_pct"
    ]
]


,segment,kpi,median_error_pct
1,all_rmg,service_time,12.500000
2,all_rmg,turnaround_time,6.666667
0,all_rmg,waiting_time,133.333333
7,delivery,service_time,-16.666667
8,delivery,turnaround_time,26.086957
6,delivery,waiting_time,133.333333
10,mixed,service_time,0.000000
11,mixed,turnaround_time,2.941176
9,mixed,waiting_time,133.333333
4,receive,service_time,0.000000


In [22]:
#====================
# FINAL SUMMARY TABLE
#=========================
validation_summary = evaluation[
    [
        "segment",
        "kpi",
        "wasserstein"
    ]
].copy()

validation_summary["mean_error_pct"] = (
    100
    * (
        evaluation["sim_mean"]
        - evaluation["ref_mean"]
    )
    / evaluation["ref_mean"]
)

validation_summary["median_error_pct"] = (
    100
    * (
        evaluation["sim_median"]
        - evaluation["ref_median"]
    )
    / evaluation["ref_median"]
)

validation_summary.round(2)

,segment,kpi,wasserstein,mean_error_pct,median_error_pct
1,all_rmg,service_time,4.63,38.77,12.50
2,all_rmg,turnaround_time,18.56,49.06,6.67
0,all_rmg,waiting_time,18.42,253.02,133.33
7,delivery,service_time,5.39,48.76,-16.67
8,delivery,turnaround_time,28.62,95.25,26.09
6,delivery,waiting_time,26.31,432.73,133.33
10,mixed,service_time,9.12,58.28,0.00
11,mixed,turnaround_time,23.17,55.18,2.94
9,mixed,waiting_time,18.62,233.63,133.33
4,receive,service_time,3.53,23.79,0.00
